# FLUX.1-schnell on free Colab (T4) — sample

Generate images with **FLUX.1-schnell** on a **free** Colab T4 GPU. No API key, no cost.

- Model: `black-forest-labs/FLUX.1-schnell` (Apache-2.0, **not gated** — no HF token needed)
- Fits in 16GB T4 via `enable_model_cpu_offload()`
- ~4 inference steps; a T4 render takes roughly **30–90s** per image

**Before you run:** `Runtime -> Change runtime type -> Hardware accelerator: T4 GPU`.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || print('No GPU! Set Runtime -> Change runtime type -> T4 GPU')

## 2. Install dependencies
Takes ~1–2 min the first time.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

## 3. Load FLUX.1-schnell
`enable_model_cpu_offload()` streams weights between CPU and GPU so it fits the T4's 16GB.
First run downloads ~24GB of weights (a few minutes on Colab's fast link).

In [ ]:
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16,
)
# Keep VRAM under the T4's 16GB. Comment out on a bigger GPU for full speed.
pipe.enable_model_cpu_offload()
print("Pipeline ready.")

## 4. Generate an image
Edit `prompt` and re-run this cell as many times as you like.

In [ ]:
prompt = "a cinematic wide shot of a lone cabin in the Tennessee smoky mountains at golden hour, mist in the valley, film grain"

image = pipe(
    prompt,
    guidance_scale=0.0,          # schnell is distilled; use 0.0
    num_inference_steps=4,       # schnell needs only ~4 steps
    height=1024,
    width=1024,
    max_sequence_length=256,
    generator=torch.Generator("cpu").manual_seed(0),
).images[0]

image.save("flux_sample.png")
image

## 5. (Optional) Batch a few prompts
Generate several stills at once — the kind of b-roll the `tennessee-bound` pipeline consumes.

In [ ]:
prompts = [
    "aerial drone shot of a winding mountain road through autumn forest",
    "close-up of an old acoustic guitar on a wooden porch, warm light",
    "a country music stage at night, stage lights, crowd silhouettes",
]

for i, p in enumerate(prompts):
    img = pipe(
        p, guidance_scale=0.0, num_inference_steps=4,
        height=1024, width=1024, max_sequence_length=256,
        generator=torch.Generator("cpu").manual_seed(i),
    ).images[0]
    img.save(f"broll_{i}.png")
    print(f"saved broll_{i}.png  ::  {p}")
print("Done. Download from the Files panel on the left.")

---
### Notes
- **Free tier limits:** Colab may disconnect after idle/long sessions and the T4 isn't guaranteed at peak times. Downloaded weights and outputs are wiped when the runtime resets.
- **Want higher quality?** `FLUX.1-dev` (gated — needs a free HF token + license accept) gives better results but needs ~24GB, so use a paid A100/L4 runtime or a rented 4090.
- **Next step for volume:** move this to a single RTX 4090 (~$0.40/hr) and drop `enable_model_cpu_offload()` for full speed — thousands of images per $10.